In [ ]:
# =============================================================================
# BOOTSTRAP — Đăng ký đường dẫn gốc của project vào Python
# =============================================================================
# Tại sao cần?
#   Notebook này nằm ở: strategies/combo/research/02_wf_optimizer.ipynb
#   Các module cần dùng (db_connector, backtest_engine...) nằm ở thư mục gốc.
#   Python mặc định không biết đường đến thư mục gốc → phải chỉ đường thủ công.
#
# Cách hoạt động:
#   Path().resolve()           → lấy đường dẫn tuyệt đối của thư mục hiện tại (research/)
#   .parent.parent.parent      → lên 3 cấp: research/ → combo/ → strategies/ → ROOT
#   sys.path.insert(0, ...)    → thêm ROOT vào đầu danh sách tìm kiếm module của Python

import sys
from pathlib import Path

_ROOT = Path().resolve().parent.parent.parent.parent  # research/ → combo/ → strategies/ → core_python/ → root
if str(_ROOT) not in sys.path:
    sys.path.insert(0, str(_ROOT))

# Combo Strategy — Portfolio Walk-Forward Optimizer
**Portfolio-Level Parameter Optimization | H4 | Walk-Forward**

Tìm bộ tham số tối ưu **ở cấp danh mục (portfolio)** bằng Walk-Forward Analysis.

**Mục tiêu chính:** Tìm tổ hợp tham số cho tất cả symbols sao cho **tổng equity portfolio** tối ưu nhất, thay vì tối ưu từng symbol riêng lẻ.

| Stage | Cell | Mô tả |
|-------|------|-------|
| Setup | 1-3 | Import, config WF folds + grid, tải dữ liệu vào RAM |
| Engine | 4 | Định nghĩa hàm Walk-Forward |
| Stage 1 | 5-7 | Per-symbol grid search → top-K candidates mỗi symbol |
| Stage 2 | 8A-8C | Portfolio grid search: K^N combinations → chọn winner |
| Heatmap | 8D | Biểu đồ kTP × x score (đánh dấu params được chọn) |
| OOS | 9 | Kiểm tra trên dữ liệu 2025 chưa từng thấy |
| Portfolio | 10 | Backtest toàn danh mục + equity curve |
| Apply | 11 | Ghi kết quả vào strategy_config.py |

> **Quy trình thực hiện:**
> Chạy Cell 1-5 (có thể mất 10-20 phút tối ưu per-symbol)
> → Cell 6-7: đánh giá candidates từng symbol
> → Cell 8A-8C: portfolio grid search → chọn winner
> → Cell 8D: xem heatmap
> → Cell 9-10: OOS + portfolio backtest
> → Cell 11: apply vào config (nếu hài lòng)

In [ ]:
# =============================================================================
# CELL 1 — SETUP: Nạp tất cả thư viện và module cần thiết
# =============================================================================
# Tại sao cần làm bước này đầu tiên?
#   Giống như trước khi nấu ăn, bạn cần lấy đầy đủ nguyên liệu và dụng cụ ra trước.
#   Tất cả công cụ tính toán, vẽ đồ thị, kết nối DB đều được nạp ở đây.

import warnings

warnings.filterwarnings('ignore')  # Tắt các cảnh báo không quan trọng để output gọn hơn

# ── Thư viện chuẩn của Python ─────────────────────────────────────────────────
import itertools  # Tạo tổ hợp tham số: 7×7×5×3 = 735 combos mỗi symbol
import json  # Xử lý dữ liệu dạng JSON (ít dùng trực tiếp, nhưng các module khác cần)
import time  # Đo thời gian chạy
from datetime import datetime  # Lấy thời gian hiện tại để đặt tên backup file
from pathlib import Path  # Xử lý đường dẫn file theo cách cross-platform

import matplotlib.gridspec as gridspec

# ── Thư viện vẽ đồ thị ────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np  # Tính toán số học nhanh (sqrt, mean, max...)

# ── Thư viện xử lý dữ liệu ────────────────────────────────────────────────────
import pandas as pd  # DataFrame — bảng dữ liệu 2 chiều, dùng cho OHLCV và kết quả
from matplotlib.colors import LinearSegmentedColormap  # Tạo bảng màu gradient cho heatmap

# Hiển thị đồ thị ngay trong notebook (không mở cửa sổ riêng)
%matplotlib inline
plt.rcParams['figure.dpi'] = 120      # Độ phân giải đồ thị (120 = sắc nét vừa đủ)

# ── Kết nối cơ sở dữ liệu ─────────────────────────────────────────────────────
# get_connection: mở kết nối đến SQL Server chứa dữ liệu OHLCV
# test_connection: kiểm tra xem DB có đang hoạt động không
from modules.db_connector import get_connection, test_connection

# ── Engine backtest ───────────────────────────────────────────────────────────
# load_backtest_full   : tải toàn bộ dữ liệu H4 của 1 symbol từ DB
# add_backtest_indicators: tính MA, MACD, ATR trên dữ liệu thô
# backtest_fast        : backtest siêu nhanh, chỉ trả về metrics (dùng cho optimizer)
# backtest_symbol      : backtest đầy đủ với trade log và equity curve (dùng cho portfolio test)
# calc_metrics         : tính toán các chỉ số hiệu suất từ trade log
# session_mask         : tạo bộ lọc giờ giao dịch
# detect_signals       : phát hiện tín hiệu mua/bán theo chiến lược Combo v2
from strategies.combo.core.backtest_engine import (
    add_backtest_indicators,
    backtest_fast,
    backtest_symbol,
    calc_metrics,
    detect_signals,
    load_backtest_full,
    session_mask,
)

# ── Cấu hình chiến lược ───────────────────────────────────────────────────────
# STRATEGY        : các tham số chung (risk, FTMO limits, trailing...)
# SYMBOLS         : danh sách 6 symbol với cấu hình từng cái (symbol_id, x, session...)
# get_symbol_ktp  : lấy kTP của 1 symbol (ưu tiên per-symbol override nếu có)
# get_symbol_params: lấy bộ params đầy đủ của 1 symbol (hàm mới thêm)
# get_indicator_params: lấy tham số indicator mặc định (MA, MACD, ATR period)
# strategy_summary: in tóm tắt cấu hình hiện tại
from strategies.combo.core.strategy_config import (
    STRATEGY,
    SYMBOLS,
    get_indicator_params,
    get_symbol_ktp,
    get_symbol_params,
)
from strategies.combo.core.strategy_config import (
    summary as strategy_summary,
)

# ── Giao diện (màu sắc, đồ thị) ──────────────────────────────────────────────
# SIGNAL, DARK, EQUITY_COLORS : bộ màu theo theme tối của hệ thống
# setup_dark_figure   : tạo figure với background tối
# color_pf            : tô màu ô theo giá trị PF (đỏ/xanh)
# dark_table_props    : style cho bảng (nền tối, chữ sáng)
from strategies.combo.core.theme import (
    DARK,
    EQUITY_COLORS,
    SIGNAL,
    color_pf,
    dark_table_props,
    setup_dark_axes,
    setup_dark_figure,
)

# In tóm tắt cấu hình chiến lược hiện tại để kiểm tra trước khi chạy
print(strategy_summary())

In [ ]:
# =============================================================================
# CELL 2 — CONFIG: Khai báo tất cả tham số điều khiển quá trình optimization
# =============================================================================
# Đây là cell DUY NHẤT bạn cần chỉnh sửa thủ công trước khi chạy.
# Tất cả quyết định về phạm vi tìm kiếm đều đặt ở đây.

# ── Thông số tài khoản backtest ───────────────────────────────────────────────
# Tại sao cần?
#   Backtest cần biết số vốn ban đầu để tính:
#   - Kích thước lot mỗi lệnh (risk_per_trade × equity)
#   - Khi nào chạm giới hạn FTMO

INITIAL_BALANCE        = 100_000    # Vốn ban đầu: $100,000 (tương ứng FTMO challenge)
RISK_PER_TRADE         = STRATEGY['risk_per_trade']    # 0.5% vốn mỗi lệnh
FTMO_DAILY_LIMIT       = STRATEGY['ftmo_daily_limit']  # Giới hạn lỗ ngày: 5%
FTMO_MAX_DD            = STRATEGY['ftmo_max_dd']       # Giới hạn drawdown tối đa: 10%
SLIPPAGE_PTS           = 2     # Trượt giá khi khớp lệnh: 2 điểm (thực tế luôn có)
COMMISSION_USD_PER_LOT = 3.5   # Phí giao dịch: $3.5/lot/chiều

# Đóng gói chi phí vào 1 dict để truyền vào backtest engine
_COSTS = {'slippage_pts': SLIPPAGE_PTS, 'commission_per_lot': COMMISSION_USD_PER_LOT}


# ── Walk-Forward Folds — Cấu trúc cửa sổ thời gian ──────────────────────────
# Tại sao dùng Walk-Forward thay vì backtest thẳng?
#   Nếu bạn tìm params tốt nhất trên toàn bộ 3 năm 2022-2024 rồi dùng luôn,
#   params đó chắc chắn rất đẹp vì đã "nhìn thấy tương lai" — gọi là overfitting.
#   Walk-Forward mô phỏng đúng thực tế: chỉ được dùng dữ liệu quá khứ để tìm params,
#   rồi test trên dữ liệu tương lai chưa thấy.
#
# Kiểu "Anchored Expanding" — train window ngày càng mở rộng:
#   WF1: Train 2022       → Test 2023 H1   (test xem params 2022 có work cho 2023 không)
#   WF2: Train 2022-2023H1→ Test 2023 H2   (train thêm, test tiếp)
#   WF3: Train 2022-2023  → Test 2024 H1
#   WF4: Train 2022-2024H1→ Test 2024 H2
#
# Mỗi tuple: (train_start, train_end, test_start, test_end, tên_fold)

WALK_FORWARD_FOLDS = [
    ('2022-01-01', '2022-12-31', '2023-01-01', '2023-06-30', 'WF1'),
    ('2022-01-01', '2023-06-30', '2023-07-01', '2023-12-31', 'WF2'),
    ('2022-01-01', '2023-12-31', '2024-01-01', '2024-06-30', 'WF3'),
    ('2022-01-01', '2024-06-30', '2024-07-01', '2024-12-31', 'WF4'),
]

# OOS = Out-of-Sample: dữ liệu 2025 hoàn toàn không được đụng đến trong quá trình
# optimization. Chỉ dùng ở Cell 9 để kiểm tra lần cuối xem params có thực sự tốt không.
OOS_START = '2025-01-01'
OOS_END   = '2025-12-31'


# ── Lưới tham số tìm kiếm — riêng từng symbol ────────────────────────────────
# Tại sao lưới khác nhau cho mỗi symbol?
#   x (breakout buffer) là số điểm tuyệt đối — US30 ở mức 40,000 pts và
#   US500 ở mức 5,000 pts nên khoảng noise và spread hoàn toàn khác nhau.
#   Dùng cùng 1 giá trị x cho tất cả sẽ quá rộng cho symbol này, quá chật cho symbol khác.
#   kTP, trailing, ma_period dùng ATR (đã normalize theo volatility) nên có thể
#   dùng cùng range — nhưng vẫn tìm riêng để mỗi symbol có kết quả tối ưu nhất.
#
# Ý nghĩa từng tham số:
#   ktp    : hệ số nhân ATR để tính TP. ktp=2.4 → TP = 2.4 × ATR tính từ entry.
#            Lớn hơn = TP xa hơn = khó đạt nhưng khi đạt thì lời nhiều hơn.
#   x      : buffer điểm khi đặt entry và SL.
#            entry_buy  = high của candle + x  (mua khi phá đỉnh)
#            sl_buy     = low  của candle - x  (cắt lỗ khi phá đáy)
#            Lớn hơn = ít bị trigger nhầm, nhưng SL distance lớn hơn = RR giảm.
#   trailing_activation: ngưỡng lợi nhuận tính bằng ATR để bắt đầu kéo SL theo MA.
#            = 1.0 → khi giá lời 1 ATR thì bắt đầu bảo vệ lợi nhuận bằng trailing SL.
#            Nhỏ hơn = bảo vệ sớm hơn nhưng có thể bị quét sớm.
#   ma_period: số nến để tính đường MA (Moving Average).
#            MA dùng 2 mục đích: xác định xu hướng (crossover signal) và làm SL trailing.
#            Nhỏ hơn = MA nhanh hơn, nhạy hơn nhưng nhiều tín hiệu giả hơn.

PARAM_GRID_PER_SYMBOL = {
    # US30 — Dow Jones (~40,000 pts) — x range 5-25 pts
    'US30': {
        'ktp':                 [1.8, 2.0, 2.2, 2.4, 2.6, 2.8, 3.0],
        'x':                   [5, 8, 10, 13, 15, 20, 25],
        'trailing_activation': [0.5, 0.75, 1.0, 1.25, 1.5],
        'ma_period':           [15, 20, 25],
    },
    # US100 — Nasdaq (~18,000 pts) — x range 3-13 pts (nhỏ hơn US30)
    'US100': {
        'ktp':                 [1.8, 2.0, 2.2, 2.4, 2.6, 2.8, 3.0],
        'x':                   [3, 4, 5, 7, 9, 11, 13],
        'trailing_activation': [0.5, 0.75, 1.0, 1.25, 1.5],
        'ma_period':           [15, 20, 25],
    },
    # US500 — S&P 500 (~5,000 pts) — x range 0.5-2.5 pts (rất nhỏ, dùng thập phân)
    'US500': {
        'ktp':                 [1.8, 2.0, 2.2, 2.4, 2.6, 2.8, 3.0],
        'x':                   [0.5, 0.8, 1.0, 1.3, 1.5, 2.0, 2.5],
        'trailing_activation': [0.5, 0.75, 1.0, 1.25, 1.5],
        'ma_period':           [15, 20, 25],
    },
    # UK100 — FTSE 100 (~8,000 pts) — x range 3-13 pts
    'UK100': {
        'ktp':                 [1.8, 2.0, 2.2, 2.4, 2.6, 2.8, 3.0],
        'x':                   [3, 4, 5, 7, 9, 11, 13],
        'trailing_activation': [0.5, 0.75, 1.0, 1.25, 1.5],
        'ma_period':           [15, 20, 25],
    },
    # HK50 — Hang Seng (~20,000 pts) — x range 8-30 pts (volatile hơn)
    'HK50': {
        'ktp':                 [1.8, 2.0, 2.2, 2.4, 2.6, 2.8, 3.0],
        'x':                   [8, 10, 13, 15, 20, 25, 30],
        'trailing_activation': [0.5, 0.75, 1.0, 1.25, 1.5],
        'ma_period':           [15, 20, 25],
    },
    # J225 — Nikkei (~38,000 pts) — x range 8-30 pts
    'J225': {
        'ktp':                 [1.8, 2.0, 2.2, 2.4, 2.6, 2.8, 3.0],
        'x':                   [8, 10, 13, 15, 20, 25, 30],
        'trailing_activation': [0.5, 0.75, 1.0, 1.25, 1.5],
        'ma_period':           [15, 20, 25],
    },
}


# ── Chọn symbols để chạy ──────────────────────────────────────────────────────
# OPTIMIZE_SYMBOLS = []          → chạy tất cả 6 symbols (mặc định)
# OPTIMIZE_SYMBOLS = ['US30']    → chỉ chạy US30 để test nhanh
OPTIMIZE_SYMBOLS = []
sym_list = [k for k in SYMBOLS if not OPTIMIZE_SYMBOLS or k in OPTIMIZE_SYMBOLS]


# ── In tóm tắt để kiểm tra trước khi chạy ────────────────────────────────────
print(f'WF folds  : {len(WALK_FORWARD_FOLDS)}')
print(f'OOS       : {OOS_START} → {OOS_END}')
print(f'Symbols   : {sym_list}')
print()
total_runs = 0
for sym in sym_list:
    g = PARAM_GRID_PER_SYMBOL[sym]
    # Số combo = tích của số giá trị mỗi tham số
    n    = len(g['ktp']) * len(g['x']) * len(g['trailing_activation']) * len(g['ma_period'])
    runs = n * len(WALK_FORWARD_FOLDS)  # Mỗi combo chạy qua tất cả folds
    total_runs += runs
    print(f'  {sym:<8}: {n} combos × {len(WALK_FORWARD_FOLDS)} folds = {runs:,} backtests')
print('  ──────────────────────────────')
print(f'  Total   : {total_runs:,} backtests')

In [ ]:
# =============================================================================
# CELL 3 — DB + PRE-CACHE DATA: Kết nối DB và tải dữ liệu vào RAM
# =============================================================================
# Tại sao cần pre-cache (tải trước vào RAM)?
#   Optimizer sẽ chạy ~18,000 backtests. Mỗi backtest cần dữ liệu OHLCV.
#   Nếu mỗi lần đọc từ DB (ổ cứng/mạng) thì sẽ cực kỳ chậm.
#   Tải 1 lần vào RAM → các lần sau dùng lại trong tích tắc.
#   Giống như: thay vì mỗi lần nấu ăn đi chợ 1 lần, ta mua đủ nguyên liệu 1 lần về nhà.

print('[DB] Connecting...')
# Kiểm tra kết nối DB — nếu fail thì dừng luôn, không tiếp tục
assert test_connection(), 'Cannot connect to DB'

print('Pre-loading data for all symbols...')
_DATA_CACHE = {}  # Dict lưu dữ liệu: key = tên symbol, value = DataFrame OHLCV

for sym_key in sym_list:
    cfg = SYMBOLS[sym_key]              # Lấy config của symbol này (gồm symbol_id)
    # load_backtest_full: tải toàn bộ dữ liệu H4 có trong DB cho symbol này
    # Không giới hạn ngày → lấy hết để optimizer tự cắt theo từng fold
    _DATA_CACHE[sym_key] = load_backtest_full(cfg['symbol_id'])
    print(f'  {sym_key}: {len(_DATA_CACHE[sym_key])} bars')

print('✓ Data ready')

In [ ]:
# =============================================================================
# CELL 4 — ENGINE: Định nghĩa các hàm thực hiện Walk-Forward
# =============================================================================
# Cell này không chạy optimization — chỉ định nghĩa "công cụ" để Cell 5 sử dụng.
# Giống như soạn sẵn công thức nấu ăn trước khi nấu.


def run_sym_wf_train(sym_key, date_from, date_to, fold_name):
    """
    GRID SEARCH trên cửa sổ TRAIN của 1 symbol.

    Nhiệm vụ:
        Thử tất cả các tổ hợp tham số trong PARAM_GRID_PER_SYMBOL[sym_key]
        trên khoảng thời gian [date_from → date_to] (giai đoạn train).
        Mỗi tổ hợp được chấm điểm bằng score.

    Trả về:
        DataFrame gồm tất cả combo và score tương ứng.
    """
    grid   = PARAM_GRID_PER_SYMBOL[sym_key]  # Lấy lưới tham số riêng của symbol này
    cfg    = SYMBOLS[sym_key]                 # Config của symbol (x gốc, session hours...)
    raw    = _DATA_CACHE[sym_key]             # Dữ liệu OHLCV đã tải sẵn trong RAM

    # itertools.product: tạo tất cả tổ hợp có thể của 4 tham số
    # Ví dụ: ktp=[1.8,2.0], x=[5,10], trailing=[0.5,1.0], ma=[15,20]
    # → 2×2×2×2 = 16 combos: (1.8,5,0.5,15), (1.8,5,0.5,20), (1.8,5,1.0,15)...
    combos = list(itertools.product(
        grid['ktp'], grid['x'],
        grid['trailing_activation'], grid['ma_period']
    ))

    rows = []  # Danh sách kết quả, sau này chuyển thành DataFrame

    for idx, (ktp, x, trail, ma_p) in enumerate(combos):
        # In progress mỗi 100 combo để người dùng biết đang chạy đến đâu
        if idx % 100 == 0:
            print(f'    {sym_key} {fold_name}: {idx}/{len(combos)} ({100*idx/len(combos):.0f}%)...',
                  end='\r')  # \r: ghi đè dòng hiện tại thay vì xuống dòng mới

        # Tính các indicator (MA, MACD, ATR) với ma_period của combo này
        # Mỗi combo có thể có ma_period khác nhau → phải tính lại indicator
        df_ind = add_backtest_indicators(raw, {'MA_PERIOD': ma_p})

        # Chạy backtest nhanh trên cửa sổ TRAIN với bộ params này
        # backtest_fast: phiên bản tối ưu tốc độ — không lưu trade log, chỉ trả metrics
        m = backtest_fast(
            sym_key, df_ind, cfg,
            ktp,    # Hệ số nhân ATR để tính TP
            x,      # Buffer điểm cho entry và SL (giá trị tuyệt đối, không phải x_mult)
            trail,  # Ngưỡng ATR để bắt đầu trailing SL
            date_from, date_to,   # Cửa sổ train
            INITIAL_BALANCE,
            costs=_COSTS,
        )

        # Lưu kết quả combo này
        rows.append(dict(
            symbol=sym_key, fold=fold_name,
            ktp=ktp, x=x, trailing=trail, ma_period=ma_p,
            score=round(m['score'], 4),  # Score = PF × sqrt(Return) / MaxDD
        ))

    print(f'    {sym_key} {fold_name}: {len(combos)}/{len(combos)} — done        ')
    return pd.DataFrame(rows)


def run_sym_wf_test(sym_key, ktp, x, trail, ma_p, date_from, date_to, fold_name):
    """
    VALIDATE 1 bộ params trên cửa sổ TEST của 1 symbol.

    Nhiệm vụ:
        Sau khi tìm top 20 params từ TRAIN, kiểm tra xem mỗi bộ params đó
        có hoạt động được trên dữ liệu TEST (tương lai chưa thấy) không.
        Đây là bước phân biệt params "thực sự tốt" vs params "may mắn trên train".

    Trả về:
        Dict chứa đầy đủ metrics: trades, PF, return, maxDD, score.
    """
    cfg    = SYMBOLS[sym_key]
    raw    = _DATA_CACHE[sym_key]
    df_ind = add_backtest_indicators(raw, {'MA_PERIOD': int(ma_p)})

    m = backtest_fast(
        sym_key, df_ind, cfg,
        ktp, x, trail,
        date_from, date_to,   # Cửa sổ TEST (khác với train)
        INITIAL_BALANCE,
        costs=_COSTS,
    )
    return dict(
        symbol=sym_key, fold=fold_name,
        ktp=ktp, x=x, trailing=trail, ma_period=ma_p,
        trades=m['trades'],
        pf=m['pf'],                      # Profit Factor: tổng lời / tổng lỗ
        ret=round(m['ret'], 2),          # Return %
        maxdd=round(m['maxdd'], 2),      # Maximum Drawdown %
        score=round(m['score'], 4),      # Score tổng hợp
    )


def calc_stability(group_df):
    """
    Tính STABILITY SCORE cho 1 bộ params dựa trên kết quả qua nhiều folds.

    Tại sao cần stability thay vì chỉ dùng avg_score?
        Params có avg_score cao nhưng chỉ tốt ở 1 fold (3 fold còn lại lỗ)
        kém hơn nhiều so với params avg_score thấp hơn nhưng tốt đều ở cả 4 folds.
        Stability đo tính nhất quán qua thời gian — quan trọng hơn peak performance.

    Công thức:
        Stability = avg_score × sqrt(n_folds) × bonus
        - avg_score      : điểm trung bình qua tất cả folds
        - sqrt(n_folds)  : thưởng cho params nhất quán qua nhiều folds
        - bonus = 1.2    : nếu TẤT CẢ folds đều có lãi (min_score > 0)
        - bonus = 0.8    : nếu có ít nhất 1 fold lỗ (rủi ro hơn)
    """
    n      = group_df['fold'].nunique()   # Số fold đã test
    avg_s  = group_df['score'].mean()     # Score trung bình
    min_s  = group_df['score'].min()      # Score thấp nhất (fold tệ nhất)
    bonus  = 1.2 if min_s > 0 else 0.8   # Thưởng nếu không có fold nào lỗ
    return round(avg_s * np.sqrt(n) * bonus, 4)


print('✓ Walk-Forward functions ready')

In [ ]:
# =============================================================================
# CELL 5 — RUN: Chạy toàn bộ quá trình optimization per-symbol
# =============================================================================
# ⚠ Cell này chạy LÂU — khoảng 10-20 phút tùy số symbols và tốc độ máy.
# Theo dõi tiến trình qua các dòng print hiển thị phần trăm hoàn thành.
#
# Luồng thực hiện:
#   Với mỗi symbol:
#     Với mỗi fold (WF1 → WF4):
#       1. Grid search 735 combos trên cửa sổ TRAIN → lấy top 20
#       2. Validate top 20 trên cửa sổ TEST → lưu kết quả
#
# Tại sao chỉ validate top 20 (không phải tất cả 735)?
#   Chạy 735 × 4 folds = 2,940 test backtests mỗi symbol là dư thừa.
#   Params xếp hạng thấp trên train rất ít khả năng tốt trên test.
#   Top 20 đủ để tìm params tốt mà không lãng phí thời gian.

started = datetime.now()
print(f'Start: {started.strftime("%H:%M:%S")}')
print('='*60)

all_train_records = []  # Tổng hợp kết quả train của tất cả symbols và folds
all_test_records  = []  # Tổng hợp kết quả test (validation) của top 20 params

for sym_key in sym_list:
    print(f'\n[{sym_key}]')  # Thông báo đang chuyển sang symbol mới

    for fold in WALK_FORWARD_FOLDS:
        train_from, train_end, test_from, test_end, fold_name = fold
        print(f'  TRAIN {train_from} → {train_end}  |  TEST {test_from} → {test_end}')

        # BƯỚC 1: Grid search trên cửa sổ TRAIN
        # → Trả về DataFrame với tất cả 735 combos được chấm điểm
        train_df = run_sym_wf_train(sym_key, train_from, train_end, fold_name)
        all_train_records.append(train_df)

        # BƯỚC 2: Lấy top 20 combo tốt nhất theo score trên train
        top20 = train_df.nlargest(20, 'score')  # nlargest = lấy 20 hàng có score cao nhất

        # BƯỚC 3: Validate từng bộ trong top 20 trên cửa sổ TEST
        for _, row in top20.iterrows():  # Duyệt qua từng hàng trong top20
            rec = run_sym_wf_test(
                sym_key, row['ktp'], row['x'], row['trailing'],
                int(row['ma_period']), test_from, test_end, fold_name
            )
            rec['train_score'] = row['score']  # Lưu thêm score train để so sánh sau
            all_test_records.append(rec)

# Tính thời gian chạy
elapsed = (datetime.now() - started).total_seconds()
print(f'\n{"="*60}')
print(f'Done in {elapsed:.0f}s ({elapsed/60:.1f} min)')

# Ghép tất cả kết quả thành 2 DataFrame lớn
train_df_all = pd.concat(all_train_records, ignore_index=True)
test_df_all  = pd.DataFrame(all_test_records)

# Lưu ra file CSV — có thể reload lại mà không cần chạy lại Cell này
train_df_all.to_csv('02_optimizer_train.csv', index=False)
test_df_all.to_csv('02_optimizer_test.csv',  index=False)
print('Saved: 02_optimizer_train.csv, 02_optimizer_test.csv')

In [ ]:
# =============================================================================
# CELL 6 — RESULTS: Tính Stability và hiển thị top 10 params cho từng symbol
# =============================================================================
# Tại sao không dùng trực tiếp score từ test mà phải tính stability riêng?
#   Vì chúng ta có kết quả từ 4 folds cho mỗi bộ params.
#   Stability tổng hợp cả 4 folds lại để chọn params ổn định nhất,
#   không phải params may mắn tốt ở 1-2 fold.

param_cols = ['ktp', 'x', 'trailing', 'ma_period']  # Cột xác định 1 bộ params

# Stage 2 dùng Top-K candidates mỗi symbol (không chỉ Top-1)
TOP_K = 5   # 5^4 = 625 portfolio combinations ở Stage 2

stability_records = []  # Sẽ chứa stability summary cho tất cả symbols

for sym_key in sym_list:
    # Lọc kết quả test chỉ của symbol này
    sym_test = test_df_all[test_df_all['symbol'] == sym_key]

    # Nhóm theo bộ params, tính các chỉ số tổng hợp qua tất cả folds
    summary = sym_test.groupby(param_cols).agg(
        n_folds        = ('fold',  'nunique'),          # Số fold đã test bộ params này
        avg_pf         = ('pf',    'mean'),             # Profit Factor trung bình
        avg_ret        = ('ret',   'mean'),             # Return % trung bình
        avg_maxdd      = ('maxdd', 'mean'),             # MaxDD % trung bình
        avg_score      = ('score', 'mean'),             # Score trung bình
        min_score      = ('score', 'min'),              # Score thấp nhất (fold tệ nhất)
        pct_profitable = ('ret',   lambda x: (x > 0).mean() * 100),  # % folds có lãi
    ).reset_index()

    summary['symbol'] = sym_key

    # Tính stability cho từng bộ params
    # Logic: avg_score × sqrt(n_folds) × (1.2 nếu tất cả folds lãi, 0.8 nếu có fold lỗ)
    summary['stability'] = summary.apply(
        lambda r: r['avg_score'] * np.sqrt(r['n_folds']) * (1.2 if r['min_score'] > 0 else 0.8),
        axis=1
    ).round(4)

    stability_records.append(summary)

# Ghép tất cả symbols lại, sắp xếp theo symbol rồi theo stability (cao nhất lên đầu)
stability_df = pd.concat(stability_records, ignore_index=True)
stability_df = stability_df.sort_values(['symbol', 'stability'], ascending=[True, False])

# Tạo top_k_params dict — Stage 2 dùng để tạo portfolio combinations
top_k_params = {}
for sym_key in sym_list:
    top_k_params[sym_key] = (
        stability_df[stability_df['symbol'] == sym_key]
        .head(TOP_K).reset_index(drop=True)
    )

# Hiển thị bảng top-K params cho từng symbol
for sym_key in sym_list:
    top10 = stability_df[stability_df['symbol'] == sym_key].head(TOP_K)
    print(f'\n─── {sym_key} — Stage 1 Top {TOP_K} stable params ───')
    display(top10[['ktp','x','trailing','ma_period',
                   'avg_pf','avg_ret','avg_maxdd','pct_profitable','stability']]
        .style
        .map(color_pf, subset=['avg_pf'])   # Tô màu cột PF: xanh = tốt, đỏ = xấu
        .format({'ktp':'{:.1f}','x':'{:.1f}','trailing':'{:.2f}',
                 'avg_pf':'{:.2f}','avg_ret':'{:+.1f}%','avg_maxdd':'{:.1f}%',
                 'pct_profitable':'{:.0f}%','stability':'{:.3f}'})
        .set_properties(**dark_table_props())
    )

In [ ]:
# =============================================================================
# CELL 7 — STAGE 1 SUMMARY: Top-K candidates per symbol
# =============================================================================
# best_params sẽ được set bởi Stage 2 (không set ở đây nữa).

print('='*62)
print('  STAGE 1 — Top-K Candidates Per Symbol')
print('='*62)

for sym_key in sym_list:
    top_k = top_k_params[sym_key]
    print(f'\n  {sym_key}')
    print(f'  {"Rank":>4}  {"kTP":>5}  {"x":>5}  {"trail":>5}  {"MA":>4}'
          f'  {"avg_ret":>7}  {"avg_dd":>7}  {"prof%":>6}  {"stab":>7}')
    for rank, row in top_k.iterrows():
        print(f'  {rank+1:>4}  {row["ktp"]:>5.1f}  {row["x"]:>5.1f}  '
              f'{row["trailing"]:>5.2f}  {int(row["ma_period"]):>4}'
              f'  {row["avg_ret"]:>+6.1f}%  {row["avg_maxdd"]:>6.1f}%'
              f'  {row["pct_profitable"]:>5.0f}%  {row["stability"]:>7.3f}')

n_combos = TOP_K ** len(sym_list)
print()
print(f'→ Stage 2: {TOP_K}^{len(sym_list)} = {n_combos:,} portfolio combinations')
print('→ best_params sẽ được gán sau Stage 2 hoàn tất.')


---
## Stage 2 — Portfolio Grid Search

Từ **Top-K candidates** mỗi symbol (Stage 1), tìm **tổ hợp symbols** tối ưu nhất ở cấp portfolio.

| Cell | Nội dung |
|------|----------|
| 8A — Engine | Pre-compute equity logs cho K × N candidates |
| 8B — Run | Grid search K^N portfolio combinations |
| 8C — Results | Rank + chọn `best_params` theo portfolio score |
| 8D — Heatmap | Biểu đồ kTP × x score (đánh dấu winner) |

> **Objective:** `score = PF × Return / MaxDD`
> **FTMO Constraints:** MaxDD < 8%, daily loss < 4%

In [ ]:
# =============================================================================
# CELL 8A — STAGE 2 ENGINE
# =============================================================================
# Pre-compute backtest_symbol() cho mỗi Top-K candidate (20 backtests).
# Lưu (trades, eq_ts) → Stage 2 Run chỉ cần merge, không re-run backtest.

S2_FROM     = '2022-01-01'
S2_TO       = '2024-12-31'
_INIT_PER_SYM = INITIAL_BALANCE / len(sym_list)


def _precompute_candidate(sym_key, ktp_val, x_val, trail_val, ma_p):
    """
    Chạy backtest đầy đủ cho 1 candidate với params override.
    Tạm thời patch SYMBOLS[sym_key]['x'] để detect_signals dùng đúng giá trị x.
    """
    cfg   = SYMBOLS[sym_key]
    raw   = _DATA_CACHE[sym_key]
    df_ind = add_backtest_indicators(raw, {'MA_PERIOD': int(ma_p)})
    df_ind['in_window'] = (
        (df_ind.index >= pd.Timestamp(S2_FROM)) &
        (df_ind.index <= pd.Timestamp(S2_TO))
    )
    # Tạm patch x để detect_signals tính RR đúng
    _orig_x = SYMBOLS[sym_key]['x']
    SYMBOLS[sym_key]['x'] = x_val
    try:
        mask   = session_mask(df_ind, cfg['session_hours_utc'])
        p_over = {'KTP': ktp_val, 'MIN_RR': STRATEGY['min_rr']}
        df_sig = detect_signals(df_ind, mask, sym_key=sym_key, params=p_over)
    finally:
        SYMBOLS[sym_key]['x'] = _orig_x
    cfg_patch  = {**cfg, 'x': x_val}
    strat_over = {'trailing_activation': trail_val}
    trades, eq_ts = backtest_symbol(
        sym_key, df_sig, cfg_patch, _INIT_PER_SYM,
        strategy=strat_over, costs=_COSTS,
    )
    return trades, eq_ts


def calc_portfolio_metrics(combo: dict, cache: dict) -> dict:
    """
    Tính portfolio-level metrics từ pre-computed equity logs.

    combo = {'US30': (ktp,x,trail,ma_p), 'UK100': ..., ...}
    """
    all_trades, eq_series = [], []
    for sym_key, params in combo.items():
        key = (sym_key, *params)
        if key not in cache:
            return dict(score=-1.0, pf=0, ret=0, maxdd=99, max_daily_dd=99, trades=0)
        trades, eq_ts = cache[key]
        all_trades.extend(trades)
        eq_series.append(eq_ts)

    if not all_trades:
        return dict(score=0, pf=0, ret=0, maxdd=0, max_daily_dd=0, trades=0)

    # Portfolio equity = cộng gộp equity series của 4 symbols
    combined = (
        pd.concat(eq_series, axis=1)
        .ffill().bfill()
        .sum(axis=1)
    )
    init_bal = _INIT_PER_SYM * len(combo)
    ret      = (combined.iloc[-1] / init_bal - 1) * 100
    peak     = combined.cummax()
    maxdd    = ((peak - combined) / peak * 100).max()

    # Max daily loss (FTMO daily limit)
    tdf = pd.DataFrame(all_trades)
    if not tdf.empty and 'exit_time' in tdf.columns:
        tdf['date']    = pd.to_datetime(tdf['exit_time']).dt.date
        daily_pnl      = tdf.groupby('date')['pnl_usd'].sum()
        max_daily_loss = abs(daily_pnl.min()) / init_bal * 100
    else:
        max_daily_loss = 0.0

    # Profit Factor
    wins = tdf[tdf['pnl_usd'] > 0]['pnl_usd'].sum() if not tdf.empty else 0
    loss = abs(tdf[tdf['pnl_usd'] < 0]['pnl_usd'].sum()) if not tdf.empty else 0
    pf   = wins / loss if loss > 0 else (9.9 if wins > 0 else 0)

    # Portfolio score — dùng sqrt(ret) nhất quán với Stage 1 score formula
    # Loại ngay nếu vi phạm FTMO
    if   maxdd        > 8.0:  score = -2.0
    elif max_daily_loss > 4.0: score = -1.0
    elif len(all_trades) < 8:  score =  0.0
    else:                      score = round(pf * np.sqrt(max(ret, 0)) / max(maxdd, 1), 4)

    return dict(
        score=round(score, 4), pf=round(min(pf, 9.9), 2),
        ret=round(ret, 2), maxdd=round(maxdd, 2),
        max_daily_dd=round(max_daily_loss, 2), trades=len(all_trades),
    )


print('Stage 2 Engine ready')
print(f'  Init per symbol : ${_INIT_PER_SYM:,.0f}')
print(f'  Date range      : {S2_FROM} to {S2_TO}')
print(f'  Pre-computations: {TOP_K} x {len(sym_list)} = {TOP_K * len(sym_list)}')
print(f'  Portfolio combos: {TOP_K}^{len(sym_list)} = {TOP_K**len(sym_list):,}')


In [ ]:
# =============================================================================
# CELL 8B — STAGE 2 RUN
# =============================================================================
# Bước 1: Pre-compute equity logs (20 backtests)
# Bước 2: Portfolio grid search (625 combinations)

from itertools import product as iproduct

# ── Bước 1: Pre-compute ────────────────────────────────────────────────────
print('Step 1: Pre-computing candidate equity logs...')
print('='*60)
s2_cache = {}
t0 = datetime.now()

for sym_key in sym_list:
    for rank, row in top_k_params[sym_key].iterrows():
        kv = float(row['ktp'])
        xv = float(row['x'])
        tv = float(row['trailing'])
        mv = int(row['ma_period'])
        key = (sym_key, kv, xv, tv, mv)
        print(f'  {sym_key} rank {rank+1}: ktp={kv} x={xv} trail={tv} ma={mv}...', end=' ')
        trades, eq_ts = _precompute_candidate(sym_key, kv, xv, tv, mv)
        s2_cache[key] = (trades, eq_ts)
        print(f'{len(trades)} trades')

print(f'Step 1 done in {(datetime.now()-t0).total_seconds():.1f}s')

# ── Bước 2: Portfolio grid search ─────────────────────────────────────────
print()
print('Step 2: Portfolio grid search...')
print('='*60)

sym_candidates = {
    s: [(float(r['ktp']), float(r['x']), float(r['trailing']), int(r['ma_period']))
        for _, r in top_k_params[s].iterrows()]
    for s in sym_list
}
all_combos = list(iproduct(*[sym_candidates[s] for s in sym_list]))
print(f'Total: {len(all_combos)} combinations')

s2_results = []
t1 = datetime.now()
for i, combo_tuple in enumerate(all_combos):
    if i % 50 == 0:
        print(f'  {i}/{len(all_combos)} ({100*i/len(all_combos):.0f}%)...', end='\r')
    combo = {sym_list[j]: combo_tuple[j] for j in range(len(sym_list))}
    m = calc_portfolio_metrics(combo, s2_cache)
    row = {'combo_id': i}
    for sym_key in sym_list:
        kv, xv, tv, mv = combo[sym_key]
        row.update({f'{sym_key}_ktp': kv, f'{sym_key}_x': xv,
                    f'{sym_key}_trail': tv, f'{sym_key}_ma': mv})
    row.update(m)
    s2_results.append(row)

s2_df = pd.DataFrame(s2_results).sort_values('score', ascending=False)
s2_df.to_csv('02_stage2_portfolio.csv', index=False)

print(f'Step 2 done in {(datetime.now()-t1).total_seconds():.1f}s')
print(f'Valid (score>0): {(s2_df["score"]>0).sum()}')
print(f'FTMO-safe (maxdd<8%): {((s2_df["score"]>0)&(s2_df["maxdd"]<8)).sum()}')
print('Saved: 02_stage2_portfolio.csv')


In [ ]:
# =============================================================================
# CELL 8C — STAGE 2 RESULTS: Chọn best_params từ portfolio winner
# =============================================================================

# ── Top 10 portfolio combinations ─────────────────────────────────────────
top10_port = s2_df[s2_df['score'] > 0].head(10)

print('='*62)
print('  STAGE 2 — Top 10 Portfolio Combinations')
print('='*62)
print(f'  {"#":>3}  {"score":>7}  {"pf":>5}  {"ret":>7}  '
      f'{"maxdd":>7}  {"daily":>7}  {"trades":>6}')
print('  ' + '-'*55)
for rank, (_, row) in enumerate(top10_port.iterrows(), 1):
    print(f'  {rank:>3}  {row["score"]:>7.4f}  {row["pf"]:>5.2f}  '
          f'{row["ret"]:>+6.1f}%  {row["maxdd"]:>6.1f}%  '
          f'{row["max_daily_dd"]:>6.1f}%  {int(row["trades"]):>6}')

# ── Gán best_params từ Stage 2 winner ─────────────────────────────────────
winner = s2_df[s2_df['score'] > 0].iloc[0]
best_params = {
    sym_key: {
        'ktp':                 float(winner[f'{sym_key}_ktp']),
        'x':                   float(winner[f'{sym_key}_x']),
        'trailing_activation': float(winner[f'{sym_key}_trail']),
        'ma_period':           int(winner[f'{sym_key}_ma']),
    }
    for sym_key in sym_list
}

print()
print('='*62)
print('  STAGE 2 WINNER — best_params')
print(f'  score={winner["score"]:.4f}  PF={winner["pf"]:.2f}  '
      f'ret={winner["ret"]:+.1f}%  maxdd={winner["maxdd"]:.1f}%  '
      f'daily={winner["max_daily_dd"]:.1f}%')
print('='*62)
print(f'  {"Symbol":<8}  {"kTP":>5}  {"x":>5}  {"trailing":>8}  {"MA":>4}')
for sym_key, bp in best_params.items():
    print(f'  {sym_key:<8}  {bp["ktp"]:>5.1f}  {bp["x"]:>5.1f}  '
          f'{bp["trailing_activation"]:>8.2f}  {bp["ma_period"]:>4}')

# ── So sánh Stage 1 vs Stage 2 ────────────────────────────────────────────
print()
print('  Stage 1 top-1 vs Stage 2 winner:')
print(f'  {"Symbol":<8}  {"S1-ktp":>7}  {"S2-ktp":>7}  {"S1-x":>6}  {"S2-x":>6}')
for sym_key in sym_list:
    s1 = top_k_params[sym_key].iloc[0]
    s2 = best_params[sym_key]
    flag = '' if (s1['ktp'] == s2['ktp'] and s1['x'] == s2['x']) else '  <- DIFFERENT'
    print(f'  {sym_key:<8}  {s1["ktp"]:>7.1f}  {s2["ktp"]:>7.1f}  '
          f'{s1["x"]:>6.1f}  {s2["x"]:>6.1f}{flag}')

print()
print('best_params ready -> Cell 9 (OOS), Cell 10 (Portfolio), Cell 11 (Apply)')


In [ ]:
# =============================================================================
# CELL 8D — HEATMAP: Biểu đồ màu kTP × x score cho từng symbol
# =============================================================================
# Tại sao cần heatmap này?
#   Heatmap giúp nhìn thấy trực quan:
#   - Vùng nào của (kTP, x) cho score cao (màu xanh/sáng)
#   - Bộ params được chọn (từ Stage 2 portfolio winner) có nằm trong vùng
#     ổn định không, hay chỉ là 1 điểm đơn lẻ may mắn (isolated peak)?
#   - Nếu vùng xung quanh best params cũng sáng màu → params đó robust.
#   - Nếu best params là 1 ô sáng duy nhất bao quanh bởi tối → nên cẩn thận.
#
# Dữ liệu: dùng train_df_all (kết quả toàn bộ grid search trên train)
# Trục X: giá trị x (breakout buffer)
# Trục Y: giá trị kTP
# Màu ô : avg score của combo (kTP, x) đó, tính trung bình qua trailing và ma_period
# Ô viền vàng: đánh dấu bộ params được chọn bởi Stage 2 (portfolio winner)

n_sym  = len(sym_list)
n_cols = 2                        # 2 cột: hiển thị 2 symbol cạnh nhau
n_rows = (n_sym + 1) // n_cols    # Số hàng cần (ví dụ 4 symbols → 2 hàng)

fig, axes = setup_dark_figure(n_rows, n_cols, figsize=(16, 5 * n_rows))
axes_flat = axes.flatten() if hasattr(axes, 'flatten') else [axes]  # Chuyển về mảng 1D

# Bảng màu: đen (score=0) → xanh đậm → xanh sáng (score cao)
cmap = LinearSegmentedColormap.from_list('bg', [DARK['bg'], '#0C447C', SIGNAL['buy']], N=256)

for idx, sym_key in enumerate(sym_list):
    ax = axes_flat[idx]

    # Lọc dữ liệu train của symbol này, chỉ lấy combo có score > 0
    sym_train = train_df_all[
        (train_df_all['symbol'] == sym_key) &
        (train_df_all['score'] > 0)
    ]
    if sym_train.empty:
        ax.set_title(f'{sym_key} — no data', color=DARK['text'])
        continue

    # Tạo pivot table: hàng = kTP, cột = x, giá trị = avg score
    # (Trung bình qua tất cả trailing và ma_period để rút gọn xuống 2D)
    hmap = sym_train.groupby(['ktp', 'x'])['score'].mean().unstack(fill_value=0)
    vmax = hmap.values.max() or 1

    # Vẽ heatmap
    im = ax.imshow(hmap.values, aspect='auto', cmap=cmap, vmin=0, vmax=vmax)

    # Nhãn trục
    ax.set_xticks(range(len(hmap.columns)))
    ax.set_xticklabels([str(v) for v in hmap.columns], color=DARK['text'], fontsize=7)
    ax.set_yticks(range(len(hmap.index)))
    ax.set_yticklabels([str(v) for v in hmap.index], color=DARK['text'], fontsize=7)
    ax.set_xlabel('x (breakout buffer)', color=DARK['text'], fontsize=8)
    ax.set_ylabel('kTP', color=DARK['text'], fontsize=8)
    ax.set_title(f'{sym_key} — avg score: kTP × x', color=DARK['text'],
                 fontsize=10, fontweight='bold')

    # In số score vào từng ô
    for yi in range(len(hmap.index)):
        for xi in range(len(hmap.columns)):
            val = hmap.values[yi, xi]
            if val > 0:
                fg = DARK['text'] if val > vmax * 0.5 else '#555555'
                ax.text(xi, yi, f'{val:.2f}', ha='center', va='center',
                        fontsize=6, color=fg)

    # Đánh dấu bộ params được chọn bởi Stage 2 bằng viền vàng
    # best_params đã được gán ở Cell 8C từ portfolio winner
    bp = best_params[sym_key]
    bk, bx = bp['ktp'], bp['x']
    if bk in list(hmap.index) and bx in list(hmap.columns):
        yi = list(hmap.index).index(bk)
        xi = list(hmap.columns).index(bx)
        ax.add_patch(plt.Rectangle((xi-.5, yi-.5), 1, 1,
                                   fill=False, edgecolor=SIGNAL['ma_line'], linewidth=2.5))

    # Thanh màu chú giải
    cb = plt.colorbar(im, ax=ax, fraction=0.025, pad=0.01)
    cb.ax.tick_params(labelcolor=DARK['text'], labelsize=6)

# Ẩn các ô thừa nếu số symbols lẻ
for i in range(n_sym, len(axes_flat)):
    axes_flat[i].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# CELL 9 — OOS VALIDATION: Kiểm tra bộ params tốt nhất trên dữ liệu 2025
# =============================================================================
# Tại sao cần OOS (Out-of-Sample) riêng biệt?
#   Dù Walk-Forward đã có test window, nhưng params vẫn được "chọn" dựa trên
#   kết quả test 2023-2024. Nghĩa là 2023-2024 vẫn gián tiếp ảnh hưởng đến quyết định.
#   Dữ liệu 2025 hoàn toàn chưa được nhìn đến trong suốt quá trình → đây mới là
#   bài kiểm tra thực sự khách quan nhất.
#
# Tiêu chí đánh giá:
#   ✓ Robust     : PF ≥ 1.3 VÀ profitable_folds ≥ 75%  → tự tin deploy
#   ~ Acceptable : PF ≥ 1.0                             → deploy nhưng theo dõi sát
#   ⚠ Weak OOS  : PF < 1.0                             → params bị overfit, cần xem lại

print(f'OOS: {OOS_START} → {OOS_END}')
print('='*60)

oos_results = {}  # Lưu metrics OOS của từng symbol
verdicts    = {}  # Lưu kết luận (Robust / Acceptable / Weak)

for sym_key in sym_list:
    bp  = best_params[sym_key]   # Bộ params tốt nhất của symbol này
    cfg = SYMBOLS[sym_key]
    raw = _DATA_CACHE[sym_key]

    # Tính indicator với ma_period của bộ params được chọn
    df_ind = add_backtest_indicators(raw, {'MA_PERIOD': bp['ma_period']})

    # Chạy backtest trên cửa sổ OOS (2025)
    m = backtest_fast(
        sym_key, df_ind, cfg,
        bp['ktp'], bp['x'], bp['trailing_activation'],
        OOS_START, OOS_END,
        INITIAL_BALANCE,
        costs=_COSTS,
    )
    oos_results[sym_key] = m

    # Lấy pct_profitable từ walk-forward results để hỗ trợ đánh giá verdict
    stab = stability_df[
        (stability_df['symbol']   == sym_key) &
        (stability_df['ktp']      == bp['ktp']) &
        (stability_df['x']        == bp['x']) &
        (stability_df['trailing'] == bp['trailing_activation']) &
        (stability_df['ma_period']== bp['ma_period'])
    ]
    pct_prof = stab['pct_profitable'].iloc[0] if not stab.empty else 0

    # Đánh giá verdict
    if m['pf'] >= 1.3 and pct_prof >= 75:
        verdict = '✓ Robust'
    elif m['pf'] >= 1.0:
        verdict = '~ Acceptable — monitor'
    else:
        verdict = '⚠ Weak OOS — possible overfit'
    verdicts[sym_key] = verdict

    print(f'  {sym_key:<8}  trades={m["trades"]:>3}  PF={m["pf"]:.2f}  '
          f'ret={m["ret"]:+.1f}%  dd={m["maxdd"]:.1f}%  {verdict}')

print('='*60)
robust_count = sum(1 for v in verdicts.values() if v.startswith('✓'))
print(f'Robust symbols: {robust_count}/{len(sym_list)}')
print()
print('Gợi ý: Symbol nào ⚠ nên xem lại grid hoặc loại khỏi danh mục.')

In [ ]:
# =============================================================================
# CELL 10 — PORTFOLIO BACKTEST: Kiểm tra toàn bộ danh mục với params per-symbol
# =============================================================================
# Tại sao cần bước này dù đã có OOS từng symbol?
#   Per-symbol OOS (Cell 9) dùng backtest_fast() — nhanh nhưng mỗi symbol chạy
#   độc lập với toàn bộ vốn $100,000.
#   Thực tế: vốn chia đều cho 6 symbols ($16,667 mỗi symbol).
#   Quan trọng hơn: cần kiểm tra các rủi ro chỉ xuất hiện ở cấp portfolio:
#     - FTMO daily limit 5% tính trên TỔNG tài khoản, không phải từng symbol.
#       → 3 symbols cùng thua 1 ngày có thể vượt giới hạn dù từng symbol vẫn OK.
#     - Correlated drawdown: US30, US100, US500 thường move cùng chiều.
#       → Worst case: cả 3 cùng hit SL 1 ngày.
#     - Equity curve tổng thể có mượt không hay quá nhiều biến động?
#
# Cell này dùng backtest_symbol() đầy đủ (có trade log) thay vì backtest_fast()
# để lấy được equity curve và tính Sharpe, Calmar ratio chính xác hơn.

BT_FROM = '2022-01-01'   # In-sample: toàn bộ giai đoạn đã dùng trong optimization
BT_TO   = '2024-12-31'

print(f'Portfolio Backtest: {BT_FROM} → {BT_TO}')
print(f'OOS check         : {OOS_START} → {OOS_END}')
print('='*60)

# Chia vốn đều cho các symbols
per_sym_bal  = INITIAL_BALANCE / len(sym_list)  # Ví dụ: 100,000 / 6 ≈ 16,667
port_trades  = []   # Tổng hợp toàn bộ lệnh của portfolio
port_equity  = {}   # Equity curve từng symbol
port_metrics = {}   # Metrics từng symbol

for sym_key in sym_list:
    bp  = best_params[sym_key]
    cfg = SYMBOLS[sym_key]
    raw = _DATA_CACHE[sym_key]

    # Tính indicator với ma_period được optimize cho symbol này
    df_ind = add_backtest_indicators(raw, {'MA_PERIOD': bp['ma_period']})

    # Đánh dấu cửa sổ backtest (chỉ lấy bars trong BT_FROM → BT_TO)
    df_ind['in_window'] = (
        (df_ind.index >= pd.Timestamp(BT_FROM)) &
        (df_ind.index <= pd.Timestamp(BT_TO))
    )

    # Phát hiện tín hiệu với kTP per-symbol
    # p_over: override params mặc định bằng kTP đã optimize cho symbol này
    hours  = cfg['session_hours_utc']
    mask   = session_mask(df_ind, hours)  # Lọc giờ giao dịch hợp lệ
    p_over = {'KTP': bp['ktp'], 'MIN_RR': STRATEGY['min_rr']}
    df_sig = detect_signals(df_ind, mask, sym_key=sym_key, params=p_over)

    # Chạy backtest đầy đủ với trailing_activation per-symbol
    # strat_over: override trailing_activation trong STRATEGY bằng giá trị đã optimize
    strat_over = {'trailing_activation': bp['trailing_activation']}
    trades, eq_ts = backtest_symbol(
        sym_key, df_sig, cfg,
        per_sym_bal,          # Vốn phân bổ cho symbol này
        strategy=strat_over,  # Override trailing activation
        costs=_COSTS,
    )

    # Tính metrics đầy đủ (PF, WR, Sharpe, Calmar...)
    m = calc_metrics(trades, eq_ts)
    port_trades.extend(trades)        # Gộp lệnh vào pool chung
    port_equity[sym_key]  = eq_ts
    port_metrics[sym_key] = m

    if m:
        print(f'  {sym_key:<8}  trades={m["total_trades"]:>3}  '
              f'WR={m["win_rate"]:>5.1f}%  PF={m["profit_factor"]:.2f}  '
              f'ret={m["total_return"]:+.1f}%  dd={m["max_drawdown"]:.1f}%')
    else:
        print(f'  {sym_key:<8}  no trades')

# ── Tổng hợp equity portfolio ─────────────────────────────────────────────────
# Ghép tất cả equity series lại, cộng tổng theo từng thời điểm
all_eq = pd.concat(port_equity.values()).sort_index()
combined_equity = all_eq.groupby(all_eq.index).sum()

# In summary portfolio
pf_vals  = [m['profit_factor'] for m in port_metrics.values() if m]
ret_vals = [m['total_return']  for m in port_metrics.values() if m]
dd_vals  = [m['max_drawdown']  for m in port_metrics.values() if m]
print('─'*60)
print(f'  Portfolio avg PF     : {np.mean(pf_vals):.2f}   ← Trung bình qua tất cả symbols')
print(f'  Portfolio avg Return : {np.mean(ret_vals):+.1f}%')
print(f'  Portfolio max DD     : {max(dd_vals):.1f}%    ← Symbol có DD lớn nhất')
print(f'  Total trades         : {len(port_trades)}      ← Tổng lệnh toàn portfolio')

# ── Vẽ equity curve ───────────────────────────────────────────────────────────
# Mỗi symbol 1 đường mờ, đường portfolio tổng 1 đường đậm
# Vùng tô đỏ = drawdown (khoảng cách từ equity xuống đỉnh cũ)
fig, ax = setup_dark_figure(figsize=(16, 5))
peak = combined_equity.cummax()  # Đỉnh cao nhất từ trước đến nay
ax.fill_between(combined_equity.index, combined_equity, peak,
                where=(combined_equity < peak), alpha=0.2, color=SIGNAL['sl'],
                label='Drawdown')
for i, (sym_key, eq) in enumerate(port_equity.items()):
    clr = EQUITY_COLORS[i % len(EQUITY_COLORS)]
    ax.plot(eq.index, eq, linewidth=0.7, alpha=0.5, color=clr, label=sym_key)
ax.plot(combined_equity.index, combined_equity,
        color=SIGNAL['ma_line'], linewidth=1.5, label='Portfolio Total')
ax.set_title('Portfolio Equity Curve — Per-Symbol Optimized Params',
             color=DARK['text'], fontsize=11, fontweight='bold')
ax.legend(fontsize=7, labelcolor=DARK['text'])
ax.set_facecolor(DARK['bg'])
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# CELL 11 — APPLY: Ghi bộ params tốt nhất vào strategy_config.py
# =============================================================================
# ⚠ CELL NÀY GHI VÀO FILE CONFIG THỰC — chỉ chạy sau khi đã review kỹ:
#   - Cell 9: OOS từng symbol — có ít nhất 4/6 symbols Robust không?
#   - Cell 10: Portfolio backtest — PF > 1.3, MaxDD < 8% không?
#
# Sau khi ghi, toàn bộ hệ thống sẽ dùng params mới:
#   - 01_backtest.ipynb (verify)
#   - signal_scanner (live signals)
#   - Telegram bot (alerts)
#
# Cơ chế ghi:
#   1. Backup file cũ → đặt tên theo timestamp (an toàn nếu muốn rollback)
#   2. Đọc nội dung file config dưới dạng text
#   3. Dùng regex để tìm và thay thế từng giá trị tham số trong text
#   4. Validate Python syntax trước khi ghi (ast.parse) → đảm bảo không làm hỏng file
#   5. Ghi file mới
#
# Nếu symbol chưa có field (vd: 'ktp') trong config → tự động THÊM VÀO.
# Nếu symbol đã có field → UPDATE giá trị cũ.

import ast
import re
import shutil

# Đường dẫn đến strategy_config.py (từ research/ lên core/)
cfg_path = Path('..') / 'core' / 'strategy_config.py'
backup   = cfg_path.parent / f'strategy_config_backup_{datetime.now().strftime("%Y%m%d_%H%M%S")}.py'

# BƯỚC 1: Backup file hiện tại
shutil.copy(cfg_path, backup)
print(f'✓ Backup: {backup.name}')

# BƯỚC 2: Đọc nội dung file config
src = cfg_path.read_text(encoding='utf-8')

# BƯỚC 3: Cập nhật từng symbol
for sym_key, bp in best_params.items():

    # --- Cập nhật x ---
    # Tìm pattern: "US30": { ... "x": <số> và thay bằng giá trị mới
    src = re.sub(
        rf'("{sym_key}":\s*{{[^}}]*?"x":\s*)([0-9.]+)',
        lambda m, v=bp['x']: m.group(1) + str(v),
        src, count=1, flags=re.DOTALL
    )

    # --- Cập nhật hoặc thêm ktp per-symbol ---
    if re.search(rf'"{sym_key}":\s*{{[^}}]*?"ktp"', src, re.DOTALL):
        # Field 'ktp' đã có → update
        src = re.sub(
            rf'("{sym_key}":\s*{{[^}}]*?"ktp":\s*)([0-9.]+)',
            lambda m, v=bp['ktp']: m.group(1) + str(v),
            src, count=1, flags=re.DOTALL
        )
    else:
        # Field 'ktp' chưa có → thêm ngay sau dòng 'x'
        src = re.sub(
            rf'("{sym_key}":\s*{{[^}}]*?"x":\s*[0-9.]+,)',
            lambda m, v=bp['ktp']: m.group(0) + f'\n        "ktp": {v},',
            src, count=1, flags=re.DOTALL
        )

    # --- Cập nhật hoặc thêm trailing_activation per-symbol ---
    if re.search(rf'"{sym_key}":\s*{{[^}}]*?"trailing_activation"', src, re.DOTALL):
        src = re.sub(
            rf'("{sym_key}":\s*{{[^}}]*?"trailing_activation":\s*)([0-9.]+)',
            lambda m, v=bp['trailing_activation']: m.group(1) + str(v),
            src, count=1, flags=re.DOTALL
        )
    else:
        src = re.sub(
            rf'("{sym_key}":\s*{{[^}}]*?"ktp":\s*[0-9.]+,)',
            lambda m, v=bp['trailing_activation']: m.group(0) + f'\n        "trailing_activation": {v},',
            src, count=1, flags=re.DOTALL
        )

    # --- Cập nhật hoặc thêm ma_period per-symbol ---
    if re.search(rf'"{sym_key}":\s*{{[^}}]*?"ma_period"', src, re.DOTALL):
        src = re.sub(
            rf'("{sym_key}":\s*{{[^}}]*?"ma_period":\s*)([0-9]+)',
            lambda m, v=bp['ma_period']: m.group(1) + str(v),
            src, count=1, flags=re.DOTALL
        )
    else:
        src = re.sub(
            rf'("{sym_key}":\s*{{[^}}]*?"trailing_activation":\s*[0-9.]+,)',
            lambda m, v=bp['ma_period']: m.group(0) + f'\n        "ma_period": {v},',
            src, count=1, flags=re.DOTALL
        )

# BƯỚC 4: Kiểm tra syntax Python hợp lệ trước khi ghi
# ast.parse sẽ báo lỗi ngay nếu regex vô tình làm hỏng cú pháp file
ast.parse(src)

# BƯỚC 5: Ghi file
cfg_path.write_text(src, encoding='utf-8')

# In tóm tắt những gì đã thay đổi
print('\n✓ strategy_config.py updated:')
print(f'  {"Symbol":<8}  {"kTP":>5}  {"x":>6}  {"trailing":>8}  {"MA":>4}')
print(f'  {"-"*45}')
for sym_key, bp in best_params.items():
    print(f'  {sym_key:<8}  {bp["ktp"]:>5}  {bp["x"]:>6}  {bp["trailing_activation"]:>8}  {bp["ma_period"]:>4}')
print()
print('Next steps:')
print('  1. Mở 01_backtest.ipynb → Restart & Run All → xác nhận kết quả')
print('  2. Nếu OK → restart signal_scanner / Telegram bot')
print(f'  3. Để rollback: copy {backup.name} → strategy_config.py')